# Baseline 3D Reconstruction Pipeline

This notebook demonstrates the complete baseline 3D reconstruction pipeline using COLMAP Structure-from-Motion.

## Pipeline Overview

1. **Image Validation** - Check image quality and validity
2. **Image Preprocessing** - Optional resizing and normalization
3. **COLMAP SfM** - Feature extraction, matching, and sparse reconstruction
4. **Point Cloud Generation** - Convert COLMAP output to point cloud
5. **Visualization** - Visualize the reconstructed point cloud

## Requirements

- COLMAP must be installed and in PATH
- Python dependencies installed (see requirements.txt)

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np

# Add scripts directory to path
scripts_dir = Path.cwd().parent / 'scripts'
sys.path.insert(0, str(scripts_dir))

print(f"Working directory: {Path.cwd()}")
print(f"Scripts directory: {scripts_dir}")

## Configuration

Set the paths for your dataset and outputs.

In [ ]:
# Paths
DATA_DIR = Path.cwd().parent / 'data' / 'raw'
PROCESSED_DIR = Path.cwd().parent / 'data' / 'processed'
WORKSPACE_DIR = Path.cwd().parent / 'workspace'
SPARSE_OUTPUT_DIR = Path.cwd().parent / 'outputs' / 'sparse'
POINT_CLOUD_DIR = Path.cwd().parent / 'outputs' / 'point_clouds'
VISUALIZATION_DIR = Path.cwd().parent / 'outputs' / 'visualizations'

# Create directories
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
SPARSE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
POINT_CLOUD_DIR.mkdir(parents=True, exist_ok=True)
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

print("Directories created/verified")

## Step 1: Image Validation

Validate the input images before processing.

In [ ]:
from validate_images import ImageValidator

validator = ImageValidator(str(DATA_DIR))
validation_result = validator.validate()

# Save validation report
validation_report_path = SPARSE_OUTPUT_DIR / 'validation_report.json'
validator.save_report(validation_result, str(validation_report_path))

print(f"\nValidation status: {validation_result['status']}")

## Step 2: Image Preprocessing (Optional)

Preprocess images if needed (resizing, normalization). Skip if images are already suitable.

In [ ]:
# Uncomment to run preprocessing
# from preprocess_images import ImagePreprocessor

# preprocessor = ImagePreprocessor(
#     input_dir=str(DATA_DIR),
#     output_dir=str(PROCESSED_DIR),
#     max_dimension=1920,  # Optional: resize to max 1920px
#     normalize=False      # Optional: normalize intensities
# )

# preprocessing_result = preprocessor.process_all()

# # Use processed images for reconstruction
# IMAGE_DIR = PROCESSED_DIR

print("Using original images (preprocessing skipped)")
IMAGE_DIR = DATA_DIR

## Step 3: COLMAP Structure-from-Motion

Run the COLMAP pipeline for feature extraction, matching, and sparse reconstruction.

In [ ]:
from run_colmap import COLMAPRunner

colmap_runner = COLMAPRunner(
    image_dir=str(IMAGE_DIR),
    workspace_dir=str(WORKSPACE_DIR),
    output_dir=str(SPARSE_OUTPUT_DIR),
    use_gpu=True  # Set to False if GPU is not available
)

colmap_result = colmap_runner.run_pipeline()

# Save COLMAP report
colmap_report_path = SPARSE_OUTPUT_DIR / 'colmap_report.json'
colmap_runner.save_report(colmap_result, str(colmap_report_path))

print(f"\nCOLMAP status: {colmap_result['status']}")

## Step 4: Point Cloud Generation

Convert COLMAP output to an Open3D point cloud and clean it.

In [ ]:
from generate_point_cloud import PointCloudGenerator

colmap_output = SPARSE_OUTPUT_DIR / 'sparse_reconstruction'

pcd_generator = PointCloudGenerator(
    colmap_output_dir=str(colmap_output),
    output_dir=str(POINT_CLOUD_DIR)
)

pcd_result = pcd_generator.process(
    remove_outliers_flag=True,
    estimate_normals_flag=False,
    nb_neighbors=20,
    std_ratio=2.0
)

# Save point cloud report
pcd_report_path = POINT_CLOUD_DIR / 'point_cloud_report.json'
pcd_generator.save_report(pcd_result, str(pcd_report_path))

print(f"\nPoint cloud status: {pcd_result['status']}")

## Step 5: Visualization

Visualize the reconstructed point cloud.

In [ ]:
from visualize_point_cloud import PointCloudVisualizer

point_cloud_path = POINT_CLOUD_DIR / 'reconstruction.ply'

visualizer = PointCloudVisualizer(
    point_cloud_path=str(point_cloud_path),
    output_dir=str(VISUALIZATION_DIR)
)

# Interactive visualization (opens window)
# viz_result = visualizer.visualize()

# Offscreen visualization (saves screenshots without window)
viz_result = visualizer.visualize_offscreen(rotate=True)

# Save visualization report
viz_report_path = VISUALIZATION_DIR / 'visualization_report.json'
visualizer.save_report(viz_result, str(viz_report_path))

print(f"\nVisualization status: {viz_result['status']}")

## Experiment: Varying Number of Views

Test reconstruction quality with different numbers of input images (3, 4, 5, 6, 10+).

To run this experiment:
1. Create subdirectories with different numbers of images (e.g., data/3_images, data/4_images, etc.)
2. Run the pipeline for each subdirectory
3. Record the results

In [ ]:
# Example experiment structure
experiment_configs = [
    {"num_images": 3, "dir": "data/3_images"},
    {"num_images": 4, "dir": "data/4_images"},
    {"num_images": 5, "dir": "data/5_images"},
    {"num_images": 6, "dir": "data/6_images"},
    {"num_images": 10, "dir": "data/10_images"},
]

experiment_results = []

for config in experiment_configs:
    print(f"\n{'='*60}")
    print(f"Running experiment with {config['num_images']} images")
    print(f"{'='*60}")
    
    # TODO: Run pipeline for each configuration
    # This would involve:
    # 1. Validate images in config['dir']
    # 2. Run COLMAP
    # 3. Generate point cloud
    # 4. Record statistics
    
    pass

print("\nExperiment setup complete. Uncomment and adapt code to run experiments.")

## Summary

This notebook demonstrated the complete baseline 3D reconstruction pipeline using COLMAP. The pipeline produces:

- Validated input images
- Sparse 3D reconstruction from COLMAP
- Cleaned point cloud in PLY format
- Visualization screenshots

### Key Statistics

Print summary statistics from the reconstruction:

In [ ]:
print("\n" + "="*60)
print("RECONSTRUCTION SUMMARY")
print("="*60)

# Validation summary
print(f"\nValidation:")
print(f"  Total images: {validation_result['total_images']}")
print(f"  Valid images: {validation_result['valid_images']}")
print(f"  Invalid images: {validation_result['invalid_images']}")

# COLMAP summary
if colmap_result['status'] == 'success':
    print(f"\nCOLMAP SfM:")
    print(f"  Status: {colmap_result['status']}")
    print(f"  Elapsed time: {colmap_result['elapsed_time']:.2f}s")
    print(f"  Model path: {colmap_result['model_path']}")

# Point cloud summary
if pcd_result['status'] == 'success':
    print(f"\nPoint Cloud:")
    print(f"  Status: {pcd_result['status']}")
    print(f"  Points: {pcd_result['statistics']['num_points']}")
    if 'bounding_box' in pcd_result['statistics']:
        bbox = pcd_result['statistics']['bounding_box']
        print(f"  Bounding box dimensions: {bbox['dimensions']}")

# Visualization summary
if viz_result['status'] == 'success':
    print(f"\nVisualization:")
    print(f"  Status: {viz_result['status']}")
    print(f"  Screenshot: {viz_result['screenshot_path']}")

print("\n" + "="*60)